# Repair notebook

Walks the conversations JSON for an existing run and back-fills the per-flag
metric columns (`{flag}_sum` and `{flag}_by_turn` for all 7 cialdini flags
and 3 improper flags) into the matching rows of the metadata xlsx.

Use this when:
- A run was started before the metric columns existed in the schema, or
- A metadata file got out of sync with its conversation JSON for any reason.

Adds the columns to the workbook header if they are not present.
Existing values in those columns (if any) are overwritten only for rows
that have a corresponding successful conversation in the JSON store.
Pending or errored rows are left alone.

In [ ]:
import conversation_generation as cg
from collections import Counter
from tqdm.auto import tqdm

## CONFIG

Set the metadata + conversations pair you want to repair. Rerun this whole
notebook for each pair you need to fix (e.g. once for threat, once for
benign).

In [ ]:
METADATA_PATH      = '../conversations/threat_metadata.xlsx'
CONVERSATIONS_PATH = '../conversations/threat_conversations.json'

## Step 1 - ensure schema

Adds any columns from the current `METADATA_COLUMNS` that are missing
from the workbook. Reports what was added (if anything).

In [ ]:
added = cg.ensure_metadata_columns(METADATA_PATH)
if added:
    print(f'Added {len(added)} column(s):')
    for c in added:
        print(f'  - {c}')
else:
    print('No columns needed to be added.')

## Step 2 - back-fill metrics

For every row in the metadata file whose `request_id` has a matching
successful conversation in the JSON store, recompute the flag metrics
and write them to the row.

The same `extract_flag_metrics` function the live pipeline uses is
called here, so values are guaranteed consistent.

In [ ]:
rows = cg.read_metadata_xlsx(METADATA_PATH)
store = cg.load_conversation_store(CONVERSATIONS_PATH)
convs = store['conversations']

# Diagnostics before we start
status_before = Counter(r['status'] for r in rows)
print(f'Metadata rows: {len(rows)}  status: {dict(status_before)}')
print(f'Conversations in JSON: {len(convs)}')

matchable = [r for r in rows if r['request_id'] in convs]
print(f'Rows with a matching conversation: {len(matchable)}')
missing_in_json = [r for r in rows if r['status'] == 'success' and r['request_id'] not in convs]
print(f'Success rows missing a JSON entry (will be skipped): {len(missing_in_json)}')

In [ ]:
# Back-fill loop
repaired = 0
skipped_no_conv = 0
skipped_bad_shape = 0

for row in tqdm(rows, desc='Repairing rows', unit='row'):
    rid = row['request_id']
    rec = convs.get(rid)
    if rec is None:
        skipped_no_conv += 1
        continue
    conversation = rec.get('conversation')
    if not isinstance(conversation, dict):
        skipped_bad_shape += 1
        continue
    try:
        turn_count = int(row['turn_count_value'])
    except (TypeError, ValueError):
        skipped_bad_shape += 1
        continue
    metrics = cg.extract_flag_metrics(conversation, turn_count)
    cg.update_metadata_row(METADATA_PATH, rid, metrics)
    repaired += 1

print(f'Repaired:               {repaired}')
print(f'Skipped (no JSON entry): {skipped_no_conv}')
print(f'Skipped (bad shape):     {skipped_bad_shape}')

## Step 3 - sanity check

Sample one repaired row and print its metric columns.

In [ ]:
rows = cg.read_metadata_xlsx(METADATA_PATH)
succ = [r for r in rows if r['status'] == 'success']
if not succ:
    print('No successful rows yet.')
else:
    r = succ[0]
    print(f'request_id: {r["request_id"]}')
    print(f'turn_count: {r["turn_count_value"]}')
    print(f'cialdini_emphasis: {r["cialdini_emphasis"][:80]}...')
    print()
    print('cialdini metrics:')
    for f in cg.CIALDINI_FLAGS:
        print(f'  {f}_sum     = {r[f+"_sum"]}')
        print(f'  {f}_by_turn = {r[f+"_by_turn"]}')
    print()
    print('improper metrics:')
    for f in cg.IMPROPER_FLAGS:
        print(f'  {f}_sum     = {r[f+"_sum"]}')
        print(f'  {f}_by_turn = {r[f+"_by_turn"]}')